# Resonance Lattice — Fabric analytics

Reads `udf_telemetry`, `udf_hits` (written by `fabric_demo_udf.ipynb`), and `Files/.rlat-builds/*.json` (written by every UDF build/refresh call). Rebuilds four conformed dims, materialises `udf_builds`, and deploys a Direct Lake semantic model `rlat-analytics` for Power BI.

Idempotent. Re-run after a batch of demo runs, or schedule via Fabric pipeline.

In [ ]:
import base64
import json
import time
import urllib.error
import urllib.request

import polars as pl
import notebookutils


## Configuration

`UDF_BASE_URL` matches the demo notebook — used here only for `list_kms` to populate `dim_km`. `SM_NAME` is the semantic model that gets created or replaced when this cell runs.

In [ ]:
UDF_BASE_URL = (
    "https://2da36c9357b24b9ea853ce08251ae0b9.z2d.userdatafunctions"
    ".fabric.microsoft.com/v1/workspaces/2da36c93-57b2-4b9e-a853-ce08251ae0b9"
    "/userDataFunctions/47c7a93e-5d6a-47e0-8f5c-2cefa5cb08b3"
)
SCHEMA = "rlat"
SM_NAME = "rlat-analytics"
OVERWRITE_SM = True


In [ ]:
TOKEN = notebookutils.credentials.getToken("pbi")
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

lh = notebookutils.lakehouse.getWithProperties(
    notebookutils.runtime.context["defaultLakehouseName"]
)
WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]
LAKEHOUSE_NAME = lh["displayName"]
SCHEMA_ENABLED = bool(lh["properties"].get("defaultSchema"))
TABLES_ROOT = f"{lh['properties']['abfsPath']}/Tables"
SQL_ENDPOINT = (
    lh["properties"].get("sqlEndpointProperties", {}).get("connectionString")
    or lh["properties"].get("sqlConnectionString")
)
storage_options = {
    "bearer_token": notebookutils.credentials.getToken("storage"),
    "use_fabric_endpoint": "true",
}


def table_path(name: str) -> str:
    if SCHEMA_ENABLED:
        return f"{TABLES_ROOT}/{SCHEMA}/{name}"
    return f"{TABLES_ROOT}/{name}"


def invoke_udf(function_name: str, parameters: dict | None = None):
    import requests
    r = requests.post(
        f"{UDF_BASE_URL}/functions/{function_name}/invoke",
        headers=HEADERS, json=parameters or {},
    )
    r.raise_for_status()
    body = r.json()
    if body.get("status") and body["status"] != "Succeeded":
        raise RuntimeError(f"{function_name} -> {body['status']}: {body.get('errors')}")
    return body.get("output", body)


## Read facts

Append-only tables, read in full each run. Direct Lake handles the size; the dim rebuild needs the full join surface anyway.

In [ ]:
fact_query = pl.read_delta(table_path("udf_telemetry"), storage_options=storage_options)
fact_hit = pl.read_delta(table_path("udf_hits"), storage_options=storage_options)
print(f"fact_query: {len(fact_query):,} rows | fact_hit: {len(fact_hit):,} rows")


## Materialise udf_builds

Every UDF build/refresh call writes a single JSON file under `Files/.rlat-builds/`. Consolidate into a Delta table so the semantic model can measure corpus-maintenance metrics (build frequency, average duration, refresh delta volume, encoder-revision spread).

In [ ]:
import os
import tempfile

builds_dir = f"{lh['properties']['abfsPath']}/Files/.rlat-builds"
try:
    listing = notebookutils.fs.ls(builds_dir)
except Exception:
    listing = []

build_rows = []
with tempfile.TemporaryDirectory() as scratch:
    for entry in listing:
        if not entry.name.endswith(".json"):
            continue
        local_path = os.path.join(scratch, entry.name)
        notebookutils.fs.cp(entry.path, f"file://{local_path}", recurse=False)
        with open(local_path, "r", encoding="utf-8") as fh:
            build_rows.append(json.load(fh))

# Explicit schema. Without it, all-build batches infer the refresh-only
# `n_added`/`n_changed`/`n_deleted`/`n_unchanged` columns as `Null` dtype
# (and vice versa for all-refresh batches) — Delta rejects `Null` columns.
_BUILDS_ROW_SCHEMA = {
    "ts": pl.Utf8, "action": pl.Utf8, "km_name": pl.Utf8,
    "source_dir": pl.Utf8, "n_passages": pl.Int64, "n_files": pl.Int64,
    "elapsed_seconds": pl.Float64, "encoder_revision": pl.Utf8,
    "store_mode": pl.Utf8, "n_added": pl.Int64, "n_changed": pl.Int64,
    "n_deleted": pl.Int64, "n_unchanged": pl.Int64,
}

if build_rows:
    udf_builds = (
        pl.DataFrame(build_rows, schema=_BUILDS_ROW_SCHEMA)
        .with_columns([
            pl.col("ts").str.to_datetime("%Y-%m-%dT%H:%M:%S%.fZ").alias("ts"),
        ])
        .with_columns([
            pl.col("ts").dt.strftime("%Y-%m-%d").alias("date_key"),
        ])
    )
else:
    # No telemetry yet. Empty frame with the post-conversion schema so
    # downstream joins / measures don't blow up; the semantic model still
    # publishes.
    udf_builds = pl.DataFrame(schema={
        **{k: (pl.Datetime if k == "ts" else v) for k, v in _BUILDS_ROW_SCHEMA.items()},
        "date_key": pl.Utf8,
    })

udf_builds.write_delta(
    table_path("udf_builds"), mode="overwrite",
    storage_options=storage_options,
    delta_write_options={"schema_mode": "overwrite"},
)
print(f"udf_builds: {len(udf_builds):,} rows -> {table_path('udf_builds')}")


## Rebuild dims

Dims are derived from the facts on every run (overwrite mode). Single owner, idempotent.

In [ ]:
dim_query = (
    fact_query.group_by("query_hash")
    .agg([
        pl.col("query").first().alias("query_text"),
        pl.col("query").first().str.split(" ").list.len().cast(pl.Int32).alias("token_count"),
        pl.col("executed_utc").min().alias("first_seen_utc"),
        pl.col("executed_utc").max().alias("last_seen_utc"),
        pl.len().cast(pl.Int64).alias("total_calls"),
    ])
)

dim_source_file = (
    fact_hit.group_by("source_file")
    .agg([
        pl.col("executed_utc").max().alias("last_seen_utc"),
        pl.len().cast(pl.Int64).alias("total_hits"),
    ])
    .with_columns(
        pl.col("source_file").str.split("/").list.first().alias("root"),
    )
)

km_meta = pl.DataFrame(invoke_udf("list_kms")).rename({"kmName": "km_name"})
# dim_km covers every km that appears in queries OR build telemetry, so a
# KM that's been built but never queried still gets a row.
all_km_names = pl.concat([
    fact_query.select("km_name"),
    udf_builds.select("km_name") if len(udf_builds) else pl.DataFrame(schema={"km_name": pl.Utf8}),
]).unique()
dim_km = all_km_names.join(km_meta, on="km_name", how="left")

# dim_date covers every day with a query OR a build, so corpus-maintenance
# measures hit a populated dim even on days with no user traffic.
fact_date_keys = pl.concat([
    fact_query.select("date_key"),
    udf_builds.select("date_key") if len(udf_builds) else pl.DataFrame(schema={"date_key": pl.Utf8}),
])
date_min = fact_date_keys["date_key"].min()
date_max = fact_date_keys["date_key"].max()
dim_date = (
    pl.date_range(
        pl.lit(date_min).str.to_date(),
        pl.lit(date_max).str.to_date(),
        interval="1d", eager=True,
    )
    .alias("date").to_frame()
    .with_columns([
        pl.col("date").dt.strftime("%Y-%m-%d").alias("date_key"),
        pl.col("date").dt.year().cast(pl.Int32).alias("year"),
        pl.col("date").dt.quarter().cast(pl.Int32).alias("quarter"),
        pl.col("date").dt.month().cast(pl.Int32).alias("month"),
        pl.col("date").dt.day().cast(pl.Int32).alias("day"),
        pl.col("date").dt.weekday().cast(pl.Int32).alias("day_of_week"),
        pl.col("date").dt.week().cast(pl.Int32).alias("iso_week"),
        pl.col("date").dt.strftime("%Y-%m").alias("year_month"),
        (pl.col("date").dt.weekday() >= 6).alias("is_weekend"),
    ])
    .select([
        "date_key", "date", "year", "quarter", "month", "day",
        "day_of_week", "iso_week", "year_month", "is_weekend",
    ])
)

for name, df in [
    ("dim_query", dim_query),
    ("dim_source_file", dim_source_file),
    ("dim_km", dim_km),
    ("dim_date", dim_date),
]:
    df.write_delta(
        table_path(name), mode="overwrite",
        storage_options=storage_options,
        delta_write_options={"schema_mode": "overwrite"},
    )
    print(f"{name}: {len(df):,} rows -> {table_path(name)}")


## Deploy semantic model

Builds `model.bim` inline (Direct Lake mode, compatibility level 1604) and posts it to the Fabric REST API. `OVERWRITE_SM=True` replaces the existing model on every re-run.

In [ ]:
FABRIC_API = "https://api.fabric.microsoft.com/v1"


def fabric_request(method, path, body=None, *, poll=True, poll_every=3.0, poll_timeout=600):
    """Call the Fabric REST API. Polls 202 LROs to completion."""
    url = path if path.startswith("http") else f"{FABRIC_API}{path}"
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(url, data=data, method=method, headers=HEADERS)
    try:
        resp = urllib.request.urlopen(req)
    except urllib.error.HTTPError as e:
        raise RuntimeError(
            f"{method} {url} -> {e.code} {e.reason}\n"
            f"{e.read().decode(errors='replace')[:1000]}"
        )
    if resp.status == 202 and poll:
        op_url = resp.headers.get("Location")
        if not op_url:
            return None
        deadline = time.time() + poll_timeout
        while True:
            time.sleep(poll_every)
            op = json.loads(urllib.request.urlopen(
                urllib.request.Request(op_url, headers=HEADERS)
            ).read())
            status = op.get("status", "")
            if status == "Succeeded":
                try:
                    return json.loads(urllib.request.urlopen(
                        urllib.request.Request(f"{op_url}/result", headers=HEADERS)
                    ).read())
                except urllib.error.HTTPError:
                    return op
            if status == "Failed":
                raise RuntimeError(f"LRO failed: {op.get('error', op)}")
            if time.time() > deadline:
                raise TimeoutError(f"LRO timed out after {poll_timeout}s: {op}")
    body_text = resp.read()
    return json.loads(body_text) if body_text else None


def b64(text):
    return base64.b64encode(text.encode("utf-8")).decode("ascii")


POLARS_TO_BIM = {
    pl.Utf8: "string", pl.Int64: "int64", pl.Int32: "int64",
    pl.Float64: "double", pl.Float32: "double", pl.Boolean: "boolean",
    pl.Date: "dateTime",
}


def bim_dtype(pl_dtype):
    if isinstance(pl_dtype, pl.Datetime):
        return "dateTime"
    return POLARS_TO_BIM.get(pl_dtype, "string")


def _bim_column(name, pl_dtype, *, hidden=False):
    c = {"name": name, "dataType": bim_dtype(pl_dtype), "sourceColumn": name}
    if hidden:
        c["isHidden"] = True
        c["summarizeBy"] = "none"
    return c


def _bim_measure(name, expression, fmt, folder, description):
    return {
        "name":          name,
        "expression":    expression,
        "formatString":  fmt,
        "displayFolder": folder,
        "description":   description,
    }


def _bim_table(name, schema, schema_name, measures, hidden_cols):
    return {
        "name":     name,
        "columns":  [_bim_column(c, dt, hidden=(c in hidden_cols))
                     for c, dt in schema.items()],
        "measures": measures,
        "partitions": [{
            "name":   name, "mode": "directLake",
            "source": {
                "type":             "entity",
                "entityName":       name,
                "schemaName":       schema_name,
                "expressionSource": "DatabaseQuery",
            },
        }],
    }


def _bim_relationship(ft, fc, tt, tc, active=True):
    import uuid
    r = {"name": str(uuid.uuid4()),
         "fromTable": ft, "fromColumn": fc,
         "toTable":   tt, "toColumn":   tc}
    if not active:
        r["isActive"] = False
    return r


def _bim_expression(sql_endpoint, lakehouse_display_name):
    return {
        "name": "DatabaseQuery", "kind": "m",
        "expression": [
            "let",
            f"    database = Sql.Database({json.dumps(sql_endpoint)}, "
            f"{json.dumps(lakehouse_display_name)})",
            "in",
            "    database",
        ],
    }


def build_bim(sm_name, sql_endpoint, lakehouse_display_name,
              tables_specs, measures_by_table, hide_cols, schema_name,
              relationships):
    return json.dumps({
        "name":              sm_name,
        "compatibilityLevel": 1604,
        "model": {
            "culture":                          "en-US",
            "discourageImplicitMeasures":       True,
            "defaultPowerBIDataSourceVersion":  "powerBI_V3",
            "expressions":                      [_bim_expression(sql_endpoint, lakehouse_display_name)],
            "tables": [
                _bim_table(name, schema, schema_name,
                           measures_by_table.get(name, []),
                           set(hide_cols.get(name, [])))
                for name, schema in tables_specs.items()
            ],
            "relationships": [_bim_relationship(*r) for r in relationships],
            "cultures":      [{"name": "en-US"}],
        },
    }, ensure_ascii=False, indent=2)


def pbism_root():
    return json.dumps({"version": "1.0", "settings": {}})


In [ ]:
TABLES = {
    "udf_telemetry":   fact_query.schema,
    "udf_hits":        fact_hit.schema,
    "udf_builds":      udf_builds.schema,
    "dim_query":       dim_query.schema,
    "dim_source_file": dim_source_file.schema,
    "dim_km":          dim_km.schema,
    "dim_date":        dim_date.schema,
}

# IDs hidden so a user dragging them onto a card doesn't get SUM(idx).
HIDE = {
    "udf_telemetry": ["query_hash", "executed_utc", "date_key"],
    "udf_hits":      ["query_hash", "executed_utc", "date_key", "passage_idx", "rank"],
    "udf_builds":    ["ts", "date_key"],
    "dim_query":     ["query_hash"],
    "dim_km":        ["km_name"],
    "dim_source_file": ["source_file"],
    "dim_date":      ["date_key"],
}

# fact -> dim (one active per fact-dim pair).
RELATIONSHIPS = [
    ("udf_telemetry", "query_hash", "dim_query",       "query_hash", True),
    ("udf_telemetry", "km_name",    "dim_km",          "km_name",    True),
    ("udf_telemetry", "date_key",   "dim_date",        "date_key",   True),
    ("udf_hits",      "query_hash", "dim_query",       "query_hash", True),
    ("udf_hits",      "km_name",    "dim_km",          "km_name",    True),
    ("udf_hits",      "date_key",   "dim_date",        "date_key",   True),
    ("udf_hits",      "source_file","dim_source_file", "source_file",True),
    ("udf_builds",    "km_name",    "dim_km",          "km_name",    True),
    ("udf_builds",    "date_key",   "dim_date",        "date_key",   True),
]

MEASURES_BY_TABLE = {
    "udf_telemetry": [
        _bim_measure(
            "Total Calls", "COUNTROWS ( 'udf_telemetry' )",
            "#,0", "01 Volume", "Total UDF calls in the selected period.",
        ),
        _bim_measure(
            "Refused Calls",
            "CALCULATE ( [Total Calls], KEEPFILTERS ( 'udf_telemetry'[refused] = TRUE() ) )",
            "#,0", "02 Refusals", "Calls that returned no hits.",
        ),
        _bim_measure(
            "Refusal Rate",
            "DIVIDE ( [Refused Calls], [Total Calls] )",
            "0.00%", "02 Refusals", "Share of calls that returned nothing.",
        ),
        _bim_measure(
            "Cold Calls",
            "CALCULATE ( [Total Calls], KEEPFILTERS ( 'udf_telemetry'[cold] = TRUE() ) )",
            "#,0", "05 Latency", "Calls that paid the cache-miss cost.",
        ),
        _bim_measure(
            "Cold Rate",
            "DIVIDE ( [Cold Calls], [Total Calls] )",
            "0.00%", "05 Latency", "Cold-start fraction; high = cache too small or too churny.",
        ),
        _bim_measure(
            "Avg Latency ms",
            "AVERAGE ( 'udf_telemetry'[latency_ms] )",
            "#,0", "05 Latency", "Mean call latency in milliseconds.",
        ),
        _bim_measure(
            "p50 Latency ms",
            "PERCENTILE.INC ( 'udf_telemetry'[latency_ms], 0.5 )",
            "#,0", "05 Latency", "Median call latency.",
        ),
        _bim_measure(
            "p95 Latency ms",
            "PERCENTILE.INC ( 'udf_telemetry'[latency_ms], 0.95 )",
            "#,0", "05 Latency", "95th-percentile call latency — the SLO line.",
        ),
        _bim_measure(
            "Cold p50 ms",
            "CALCULATE ( [p50 Latency ms], KEEPFILTERS ( 'udf_telemetry'[cold] = TRUE() ) )",
            "#,0", "05 Latency", "Median latency on cache-miss calls.",
        ),
        _bim_measure(
            "Warm p50 ms",
            "CALCULATE ( [p50 Latency ms], KEEPFILTERS ( 'udf_telemetry'[cold] = FALSE() ) )",
            "#,0", "05 Latency", "Median latency on cache-hit calls.",
        ),
        _bim_measure(
            "Avg Top1 Score",
            "AVERAGE ( 'udf_telemetry'[top1_score] )",
            "0.000", "01 Volume", "Mean rank-1 cosine across calls. Higher = stronger best hit.",
        ),
    ],
    "udf_hits": [
        _bim_measure(
            "Total Hits", "COUNTROWS ( 'udf_hits' )",
            "#,0", "03 Coverage", "Total retrieved passages across all calls.",
        ),
        _bim_measure(
            "Verified Hits",
            "CALCULATE ( [Total Hits], KEEPFILTERS ( 'udf_hits'[drift_status] = \"verified\" ) )",
            "#,0", "04 Drift", "Hits whose source bytes still match the build hash.",
        ),
        _bim_measure(
            "Verified Rate",
            "DIVIDE ( [Verified Hits], [Total Hits] )",
            "0.00%", "04 Drift", "Trust score — share of hits still matching source. Goal=100%.",
        ),
        _bim_measure(
            "Drifted Hits",
            "CALCULATE ( [Total Hits], KEEPFILTERS ( 'udf_hits'[drift_status] <> \"verified\" ) )",
            "#,0", "04 Drift", "Hits where the source has changed since build — rebuild signal.",
        ),
    ],
    "udf_builds": [
        _bim_measure(
            "Builds Run", "COUNTROWS ( 'udf_builds' )",
            "#,0", "06 Maintenance", "Total build + refresh invocations in the selected period.",
        ),
        _bim_measure(
            "Refresh Calls",
            "CALCULATE ( [Builds Run], KEEPFILTERS ( 'udf_builds'[action] = \"refresh\" ) )",
            "#,0", "06 Maintenance", "Incremental delta-apply calls (action=refresh).",
        ),
        _bim_measure(
            "Build Calls",
            "CALCULATE ( [Builds Run], KEEPFILTERS ( 'udf_builds'[action] = \"build\" ) )",
            "#,0", "06 Maintenance", "Full-build calls (action=build) — first-time or fall-back path.",
        ),
        _bim_measure(
            "Avg Build Duration s",
            "AVERAGE ( 'udf_builds'[elapsed_seconds] )",
            "#,0.0", "06 Maintenance",
            "Mean wall-clock seconds per build/refresh call. Tracks corpus growth + Fabric CPU.",
        ),
        _bim_measure(
            "Refresh Delta Volume",
            "SUM ( 'udf_builds'[n_added] ) + SUM ( 'udf_builds'[n_changed] )",
            "#,0", "06 Maintenance",
            "Total passages added or re-encoded by refresh in the period — corpus churn proxy.",
        ),
        _bim_measure(
            "Encoder Revisions In Fleet",
            "DISTINCTCOUNT ( 'udf_builds'[encoder_revision] )",
            "#,0", "06 Maintenance",
            "Distinct encoder revisions stamped into built .rlats. >1 = stragglers; rebuild laggards.",
        ),
    ],
}

if not SQL_ENDPOINT:
    lh_full = fabric_request(
        "GET", f"/workspaces/{WORKSPACE_ID}/lakehouses/{lh['id']}", poll=False,
    )
    SQL_ENDPOINT = lh_full["properties"]["sqlEndpointProperties"]["connectionString"]

schema_for_bim = SCHEMA if SCHEMA_ENABLED else "dbo"
model_bim = build_bim(
    SM_NAME, SQL_ENDPOINT, LAKEHOUSE_NAME,
    TABLES, MEASURES_BY_TABLE, HIDE, schema_for_bim, RELATIONSHIPS,
)

definition = {
    "parts": [
        {"path": "definition.pbism", "payload": b64(pbism_root()), "payloadType": "InlineBase64"},
        {"path": "model.bim",        "payload": b64(model_bim),    "payloadType": "InlineBase64"},
    ]
}

existing = fabric_request(
    "GET", f"/workspaces/{WORKSPACE_ID}/semanticModels", poll=False,
)
existing_id = next(
    (m["id"] for m in (existing or {}).get("value", []) if m["displayName"] == SM_NAME),
    None,
)
if existing_id and OVERWRITE_SM:
    fabric_request(
        "DELETE", f"/workspaces/{WORKSPACE_ID}/semanticModels/{existing_id}", poll=False,
    )
    existing_id = None

if existing_id:
    fabric_request(
        "POST",
        f"/workspaces/{WORKSPACE_ID}/semanticModels/{existing_id}/updateDefinition",
        {"definition": definition},
    )
    print(f"updated {SM_NAME}")
else:
    fabric_request(
        "POST",
        f"/workspaces/{WORKSPACE_ID}/semanticModels",
        {"displayName": SM_NAME, "definition": definition},
    )
    print(f"created {SM_NAME}")

print(f"open in Power BI: workspace={WORKSPACE_ID} model={SM_NAME}")


## Reports

Open `rlat-analytics` in Power BI Desktop (Get Data → Power BI semantic models). Six visuals to build:

- Top queries: `dim_query[query_text]` × `[Total Calls]`, slice by `dim_date[year_month]`
- Refusal rate: `[Refusal Rate]` line over `dim_date`, drill-through to `dim_query` filtered to refusals
- Source file Pareto: `dim_source_file[source_file]` × `[Total Hits]`
- Verified rate by km: `[Verified Rate]` × `dim_km` × `dim_date[iso_week]`
- Latency: `[p50 Latency ms]` and `[p95 Latency ms]` line over `dim_date`; `[Cold p50 ms]` vs `[Warm p50 ms]` cards
- Corpus maintenance: `[Builds Run]` + `[Avg Build Duration s]` over `dim_date[iso_week]`; `[Refresh Delta Volume]` per `dim_km`; `[Encoder Revisions In Fleet]` card — straggler signal